In [2]:
import os
import json
import glob
from collections import Counter

# ================= 配置区 =================
METADATA_DIR = "./triplets/metadata"
# =========================================

def analyze_rules():
    if not os.path.exists(METADATA_DIR):
        print(f"❌ 找不到目录: {METADATA_DIR}")
        return

    json_files = glob.glob(os.path.join(METADATA_DIR, "*.json"))
    if not json_files:
        print(f"⚠️ 在 {METADATA_DIR} 中没有找到任何 JSON 文件。")
        return

    total_triplets = 0
    valid_count = 0
    invalid_count = 0
    
    # 进一步分析：如果是 invalid，它们主要发生在哪些阶段？
    invalid_phase_counter = Counter()

    print(f"🔍 正在扫描并统计 {len(json_files)} 个切片文件的合规性 (rule_is_valid)...")

    for file_path in json_files:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                clip_data = json.load(f)
        except Exception as e:
            print(f"❌ 无法读取文件 {file_path}: {e}")
            continue

        triplets_list = clip_data.get("triplets", [])

        for item in triplets_list:
            total_triplets += 1
            
            # 读取我们写入的软标签，如果没有这个字段，默认视作 False 以防万一
            is_valid = item.get("rule_is_valid", False)
            
            if is_valid:
                valid_count += 1
            else:
                invalid_count += 1
                # 记录一下是在哪个阶段发生的不合规
                triplet = item.get("triplet", [])
                if len(triplet) == 3:
                    phase = triplet[2]
                    invalid_phase_counter[phase] += 1

    # 打印精美的统计报告
    print("\n==================================================")
    print(" 📊 Triplet 合规性 (rule_is_valid) 统计报告")
    print("==================================================")
    print(f"  总三元组数量 : {total_triplets}")
    print(f"  ✅ 合规 (True) : {valid_count} (占比: {(valid_count / total_triplets * 100):.2f}%)")
    print(f"  ❌ 冲突 (False): {invalid_count} (占比: {(invalid_count / total_triplets * 100):.2f}%)")
    print("--------------------------------------------------")
    
    if invalid_count > 0:
        print(" 🔍 冲突高发阶段 Top 5:")
        for phase, count in invalid_phase_counter.most_common(5):
            print(f"    - {phase:<25} : {count} 次")
    print("==================================================")
    print("💡 提示：此处的冲突数量应与 verify_triplets.py 扫出的错误数高度一致（如果只有工具 None 报错的话）。")

if __name__ == "__main__":
    analyze_rules()

🔍 正在扫描并统计 4391 个切片文件的合规性 (rule_is_valid)...

 📊 Triplet 合规性 (rule_is_valid) 统计报告
  总三元组数量 : 4391
  ✅ 合规 (True) : 4067 (占比: 92.62%)
  ❌ 冲突 (False): 324 (占比: 7.38%)
--------------------------------------------------
 🔍 冲突高发阶段 Top 5:
    - Hydrodissection           : 69 次
    - Capsulorhexis             : 59 次
    - Lens positioning          : 48 次
    - Irrigation_Aspiration     : 39 次
    - Phacoemulsification       : 35 次
💡 提示：此处的冲突数量应与 verify_triplets.py 扫出的错误数高度一致（如果只有工具 None 报错的话）。
